# User flows

*One section per thing a user does with the SyftHub Python SDK, in the order a new user meets them. Everything runs
against in-memory fakes that return exactly the shapes the real Hub and Space APIs return; names, prices and
documents are invented. The API is the one in `LOW_LEVEL_DESIGN.md`.*

The SDK is **async-first**: composing a search or a chat is free and synchronous, anything that touches the network
is awaited. A generated synchronous twin exists for scripts; it appears at the end.

## 1. Hello world

Install, construct, search. The Hub URL and a token come from `SYFTHUB_URL` and `SYFTHUB_TOKEN` when they are set.
Without `sources`, the search runs over **free data sources** the Hub ranks for the query, so a first call never costs
anything. Paid endpoints are never included implicitly.

In [1]:
import syfthub

hub = syfthub.AsyncHub()                                   # mock: in-memory fakes behind the scenes
results = await hub.search("adverse events in phase 3 trials").execute()
results

endpoint,returns,outcome,result,cost,latency
carol/papers,references,returned,5 documents,-,271 ms
olga/trials-assistant,both,returned,"2 documents · summary: Across my curated corpus, grade 3+ neutropenia is …",-,583 ms
frank/registry,references,rejectedrate limited,"Policy 'rate_limit' blocked request: {'limit': '2/m', 'remaining': 0} · resets in 60s",-,245 ms


Each row is one Space's answer. Rows are typed objects, so the IDE knows what is on them.

In [2]:
for path, doc in results.documents[:3]:
    print(f"{path:<16} {doc.similarity_score:.2f}  {doc.content[:70]}")

carol/papers     0.91  Grade 3+ neutropenia occurred in 41% of the treatment arm versus 18% o
carol/papers     0.87  Serious adverse events were reported in 23% of patients; most common w
carol/papers     0.80  Treatment discontinuation due to adverse events: 12.4% vs 6.1% (placeb


## 2. Sign in

Three ways, all under `hub.auth`. The one top-level alias is `hub.login`, because it is the first thing a new user
types. For the rest of this notebook we use an explicit `World` so we can stand in for the outside world: a payment
provider's webhook, a Space going offline.

In [3]:
import syfthub
from syfthub import AsyncHub, Options
from syfthub.testing import World, MockTransport

world = World()                                     # the mock's stand-in for the outside world
hub = AsyncHub(options=Options(http_client=MockTransport(world)))
await hub.login(username="alice", password="secret")

Hub wallet,"$1.50 (MPP only, experimental)"


In [4]:
hub.me

Hub wallet,"$1.50 (MPP only, experimental)"


In [5]:
await hub.auth.whoami()                                   # refreshes the Hub wallet balance; raises NotLoggedIn when anonymous

Hub wallet,"$1.50 (MPP only, experimental)"


## 3. Find sources

Discovery lives on `hub.endpoints`. `list()` follows the Hub's pages for you; `type` and `owner` are applied by the
Hub, everything else locally. The result is an `EndpointList` you can keep narrowing without a round trip.

In [6]:
eps = await hub.endpoints.list()
eps

endpoint,type,pricing,policies,about,★
bob/llama-3Llama 3 70B,model,free,,"General model, hosted by Bob. Free.",88
carol/papersOncology Trial Papers,data_source,free,,"Indexed phase 2/3 oncology publications, 2018-2026, with adverse-event tables.",42
erin/trialsTrial Registry Mirror,data_source,$0.05/request via stripe,,Structured registry records with adverse-event tables.,31
olga/trials-assistantTrials Assistant,model_data_source,free,,Answers over Olga's curated trial corpus and returns the passages it used.,27
lena/sahabat-aiSahabat-AI 8B,model,Rp300/request via xendit,,"Small bilingual model, hosted by Lena. Billed in IDR from the same wallet as her registry.",23
ivan/gpt-miniGPT Mini (metered),model,$0.02/request via stripe,,"Small fast model, prepaid per message.",23
dave/notesClinician Notes (de-identified),data_source,$0.02/document via xendit,rate_limit,De-identified progress notes from two teaching hospitals.,17
heidi/shared-corpusConsortium Corpus,data_source,$0.01/request via cluster,,Shared corpus billed through the consortium's managed wallet.,12
dave/imagingRadiology Reports (de-identified),data_source,$0.03/request via xendit,,Same wallet as dave/notes: one top-up funds both.,9
lena/id-registryIndonesian Trial Registry,data_source,Rp300/request via xendit,,Billed in IDR: a separate wallet even though the rail is xendit too.,6


In [7]:
picked = (eps.matching("trial adverse events").filter(type="data_source")
          + eps.pick("dave/notes", "olga/trials-assistant"))
picked.paths

['carol/papers',
 'erin/trials',
 'frank/registry',
 'dave/notes',
 'olga/trials-assistant']

On a Hub too large to list, ask it to search its listings semantically, or walk the pages yourself.

In [8]:
(await hub.endpoints.search("clinical notes")).paths

['dave/notes', 'dave/imaging']

In [9]:
async for page in hub.endpoints.pages(page_size=6):
    print(f"offset {page.offset}: {len(page.items)} of {page.total}")

offset 0: 6 of 14
offset 6: 6 of 14
offset 12: 2 of 14


One endpoint's record: every policy in words, pricing decoded from the payment policy, nothing invented.

In [10]:
await hub.endpoints.get("dave/notes")

Endpoint(dave/notes, data_source, $0.02/document via xendit)

## 4. Wallets

Prepaid balances live on `hub.wallets`. One wallet per owner, rail and currency; several endpoints can bill the same
wallet, so one top-up funds all of them. Amounts are `Money`: a `Decimal` with a currency, and two currencies never
add up by accident.

In [11]:
await hub.wallets.list()

wallet owner,rail,currency,your balance,backs,bundles
erinw-erin-stripe-usd,stripe,USD,$1.00,erin/trials,"starter $5.00, pro $20.00"
lenaw-lena-xendit-idr,xendit,IDR,"Rp5,000",lena/id-registrylena/sahabat-ai,"starter Rp75,000, pro Rp300,000"
ivanw-ivan-stripe-usd,stripe,USD,$0.00,ivan/gpt-mini,"starter $5.00, pro $20.00"
davew-dave-xendit-usd,xendit,USD,$0.00,dave/imagingdave/notes,"starter $5.00, pro $20.00"
consortiumw-consortium-cluster-usd,cluster,USD,$3.00,heidi/shared-corpus,"starter $5.00, pro $20.00"


In [12]:
w = await hub.wallets.for_endpoint("dave/notes")
w.balance, w.endpoints, [b.id for b in w.bundles]

(Money(0.0, 'USD'), ('dave/imaging', 'dave/notes'), ['starter', 'pro'])

## 5. Search: compose, pre-flight, execute

`hub.search(...)` composes a `SearchPlan`. Nothing is sent and nothing is awaited. Advanced users keep the plan around
and inspect it; everyone else chains straight to `execute()`.

In [13]:
plan = hub.search("What adverse events were reported in phase 3 trials?", sources=picked)
plan

SearchPlan("What adverse events were reported in phase 3 trials?", 5 sources, not pre-flighted: await plan.preflight())

`await plan.preflight()` fills the table from Hub metadata, one balance call per prepaid wallet, and the session budget.
Dave's notes are **held**: the wallet is empty. The card says what to do.

In [14]:
await plan.preflight()

SearchPlan "What adverse events were reported in phase 3 trials?"  est. max $0.05  (4 of 5 will be sent)
  carol/papers             free                             max $0.00    ready
  erin/trials              $0.05/request via stripe         max $0.05    ready
  frank/registry           free                             max $0.00    ready
  dave/notes               $0.02/document via xendit        max $0.10    needs credits  (held)
  olga/trials-assistant    free                             max $0.00    ready

Credits are bought from Dave's Space. `top_up` returns an `Invoice` with the checkout link; the provider's webhook
credits the wallet. Here `world.pay` plays the user and the webhook.

In [15]:
invoice = await plan.top_up("dave/notes", "starter")
invoice

wallet,w-dave-xendit-usd
bundle,starter · $5.00
checkout,https://checkout.example/pay/inv-33059


In [16]:
world.pay(invoice)                                        # mock: paid at the link, webhook fired
await plan.refresh()

endpoint,pricing,max cost,your balance,pre-flight,
carol/papers,free,-,-,ready,
erin/trials,$0.05/request via stripe,$0.05,$1.00,ready,deducted from wallet erin stripe USD
frank/registry,free,-,-,ready,rate limit 2/m per_user
dave/notes,$0.02/document via xendit,$0.10,$5.00,ready,deducted from wallet dave xendit USD rate limit 60/m per_user
olga/trials-assistant,free,-,-,ready,


`execute()` re-runs pre-flight (fresh balances count), then asks every source that passes, in parallel, bounded by
`Options.concurrency`. Frank's feed is rate limited; that cannot be predicted, so it was sent and came back
**rejected**, with what the Space said about its limit surfaced on the row. The charges table is assembled from the
Spaces' own receipts.

In [17]:
results = await plan.execute()
results

endpoint,returns,outcome,result,cost,latency
carol/papers,references,returned,5 documents,-,299 ms
erin/trials,references,returned,3 documents,$0.05,411 ms
frank/registry,references,rejectedrate limited,"Policy 'rate_limit' blocked request: {'limit': '2/m', 'remaining': 0} · resets in 60s",-,303 ms
dave/notes,references,returned,4 documents,$0.08,151 ms
olga/trials-assistant,both,returned,"2 documents · summary: Across my curated corpus, grade 3+ neutropenia is …",-,517 ms
source,policy,status,amount,wallet / rail,transaction
erin/trials,stripe_per_request,charged,$0.05,w-erin-stripe-usd,ledger-492951
dave/notes,xendit_per_document,charged,$0.08,w-dave-xendit-usd,ledger-544851


In [18]:
results["erin/trials"]

policy_type,status,amount,transaction / reason_code,
stripe_per_request,charged,$0.05,ledger-492951,


Results are **immutable**. Views return a new `Results` and leave the raw responses alone; combine them with `+`.
`to_dict()` gives stable JSON for a chatbot or a cache.

In [19]:
recent = results.filter(published_gte="2024-01-01")
len(results.documents), len(recent.documents), len(recent.top(3).documents)

(14, 10, 3)

In [20]:
recent.only("carol/papers", "erin/trials").top(3)

endpoint,returns,outcome,result,cost,latency
carol/papers,references,returned,2 documents of 5 returned,-,299 ms
erin/trials,references,returned,1 document of 3 returned,$0.05,411 ms
source,policy,status,amount,wallet / rail,transaction
erin/trials,stripe_per_request,charged,$0.05,w-erin-stripe-usd,ledger-492951


In [21]:
import json
snapshot = results.to_dict()
print(json.dumps(snapshot)[:200], "…")
syfthub.Results.from_dict(hub, snapshot).ok_count

{"query": "What adverse events were reported in phase 3 trials?", "limit": 5, "threshold": 0.5, "note": null, "history": [], "filters": {}, "estimate": {"USD": "0.15"}, "rows": [{"endpoint": {"name":  …


4

## 6. When a Space says no

A Space's answer is never an exception: it is a row with an outcome that says **who** stopped the call. `HELD` is
pre-flight (nothing spent), `REJECTED` is the Space's policy (403), `FAILED` is transport or server trouble after
retries, `PAYMENT_REQUIRED` is an MPP 402. `retry()` re-sends whatever can change.

In [22]:
row = results["frank/registry"]
row.outcome, row.reason, row.retryable, row.rate_limit

(<Outcome.REJECTED: 'rejected'>,
 <Reason.RATE_LIMITED: 'rate_limited'>,
 True,
 RateLimit(limit='2/m', remaining=0, reset_seconds=60))

The transport owns the retry policy. Network errors and 5xx get exponential backoff with jitter. A rate limit whose
reset is short enough is waited out inside the call; the default ceiling is 10 seconds, so Frank's 60-second window
was surfaced instead. Raise the ceiling and the same call waits and succeeds (the mock's clock runs at 100×).

In [23]:
from syfthub import Retry, CircuitBreaker

patient = AsyncHub(options=Options(http_client=MockTransport(world), retries=Retry(rate_limit_wait=90)))
await patient.login(username="alice", password="secret")
r = await patient.search("interstitial lung disease signal", sources=["frank/registry"]).execute()
r["frank/registry"].note, r["frank/registry"].attempts, r.ok_count

('waited 60s for the rate limit to reset', 2, 1)

Grace's archive is offline. Three attempts, then a `FAILED` row. After `CircuitBreaker.failures` consecutive
failures the SDK stops sending to that Space for the cooldown: the next plan holds it as `circuit_open` without a
request. Here the breaker is set to trip on the first failure so you can see it.

In [24]:
fragile = AsyncHub(options=Options(http_client=MockTransport(world), circuit_breaker=CircuitBreaker(failures=1, cooldown=60)))
first = await fragile.search("anything", sources=["grace/archive", "carol/papers"]).execute()
first

circuit open for https://grace.spaces.example after 1 failures; cooling down 60s


endpoint,returns,outcome,result,cost,latency
grace/archive,references,failedunreachable,https://grace.spaces.example: connection refused · 3 attempts,-,13 ms
carol/papers,references,returned,5 documents,-,552 ms


In [25]:
second = await fragile.search("anything", sources=["grace/archive", "carol/papers"]).execute()
second["grace/archive"]

SourceResult(grace/archive, held (circuit_open), 0 docs)

Give up on a source and `retry()` leaves it alone.

In [26]:
done = results.skip("frank/registry")
[(r.endpoint.path, r.outcome.value, r.reason.value if r.reason else None) for r in done if not r.ok]

[('frank/registry', 'held', 'by_user')]

## 7. A spending cap

`set_budget` caps the session. Pre-flight holds any source whose estimate would cross it; nothing is sent and nothing
raises. `ignore_budget=True` sends the held rows this once.

In [27]:
hub.set_budget(0.05)
capped = await hub.search("infection-related adverse events", sources=["erin/trials", "heidi/shared-corpus"]).preflight()
capped

endpoint,pricing,max cost,your balance,pre-flight,
erin/trials,$0.05/request via stripe,$0.05,$0.95,ready,deducted from wallet erin stripe USD
heidi/shared-corpus,$0.01/request via cluster,$0.01,$3.00,over budgetheld,"would bring this session to $0.06, budget is $0.05"


In [28]:
r = await capped.execute()
r2 = await r.retry(ignore_budget=True)
[(x.endpoint.path, x.outcome.value) for x in r2], hub.budget

([('erin/trials', 'returned'), ('heidi/shared-corpus', 'returned')],
 Budget($-0.01 of $0.05 left))

In [29]:
hub.budget = None                                         # lift it for the rest of the notebook

## 8. Headless: a script or a CI job

No notebook, no buttons. Sign in with a token, let the SDK log one line per fan-out, and when a wallet is short
create the invoice and **wait for the payment** by polling the balance route. Here a background task plays the
person who pays after a while.

In [30]:
import asyncio, logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

ci_world = World()
ci = AsyncHub(token="syft_pat_alice", options=Options(http_client=MockTransport(ci_world)))
async with ci:
    plan = await ci.search("phase 3 adverse events", sources=["dave/notes", "carol/papers"], limit=3).preflight()
    if plan.pending:
        invoice = await plan.top_up(plan.pending[0].wallet.key, "starter")
        print("pay at", invoice.checkout_url)

        async def someone_pays():                           # mock: the finance team clicks the link later
            await asyncio.sleep(0.3)
            ci_world.pay(invoice)
        asyncio.create_task(someone_pays())

        invoice = await invoice.wait_paid(timeout=600, poll=5)   # polls the balance; the mock clock runs fast
        print("invoice", invoice.id, invoice.status.value)

    results = await plan.execute()
    print(json.dumps({"ok": results.ok_count, "cost": {k: str(v.amount) for k, v in results.cost.items()},
                      "documents": len(results.documents)}))
logging.getLogger().setLevel(logging.WARNING)

pay at https://checkout.example/pay/inv-69852


invoice inv-69852 paid


INFO syfthub: search "phase 3 adverse events": 2 returned, 0 held, 0 rejected, 0 failed, 0 waiting · 0.4s


{"ok": 2, "cost": {"USD": "0.06"}, "documents": 6}


## 9. Talk it through: the chat room

A model is a Space endpoint, so a chat has the same pre-flight as a search. `results.chat(model)` opens a room over
a results view; pre-flight runs on the first await. Ivan's model is metered and the wallet is empty, so the first
message is held with the top-up attached, never an exception.

In [31]:
chat = recent.chat("ivan/gpt-mini")
await chat.preflight()

Chat(models=['ivan/gpt-mini'], via=direct, turns=0)

In [32]:
chat.context                                              # exactly what the next message will carry, with token counts

Context(0 earlier turns ~0 tokens, 10 references, 1 summaries ~249 tokens)

In [33]:
await chat.send("Summarise the serious adverse events across these sources.")

wallet,ivan · stripe USD w-ivan-stripe-usd
your balance,$0.00 · needed $0.02
bundles,starter: $5.00 pro: $20.00


In [34]:
invoice = await chat.top_up(bundle="starter")
world.pay(invoice)
await chat.send("Summarise the serious adverse events across these sources.")

[1],carol/papers
[2],carol/papers
[4],erin/trials


Follow-ups ride on the transcript. `use(view)` swaps the passages and keeps the turns.

In [35]:
await chat.send("Were any of them fatal?")

[3],carol/papers
[4],erin/trials


In [36]:
chat.use(recent.only("erin/trials", "olga/trials-assistant").top(3))
await chat.send("And in the registry data alone, what stood out?")

[1],erin/trials
[2],erin/trials


A room holds several models. `add` brings one in from the next turn, briefed with the transcript so far; `send` then
returns one `Reply` per model, indexable by path. `remove` stops asking one; its tab stays readable.

In [37]:
await chat.add("bob/llama-3")
replies = await chat.send("Do you agree, and what would you add?")
replies

[4],olga/trials-assistant
[4],olga/trials-assistant


In [38]:
replies["bob/llama-3"].text

'Yes, both points hold against the passages in context; the registry and the pooled analysis agree on the SAE range and the dose-limiting toxicities. I would add that hepatic events stayed under 2% [4].'

In [39]:
await chat.remove("ivan/gpt-mini")
chat

Chat(models=['bob/llama-3'], via=direct, turns=5)

The room **mutates in place**; it is the one stateful object in the SDK. To compare two views over one conversation,
`fork()` first. Replies are frozen snapshots, so an earlier turn's `context` never changes under you.

In [40]:
branch = chat.fork().only("erin/trials")
len(chat.view), len(branch.view), chat.turn == branch.turn

(2, 1, True)

In [41]:
chat.spent, chat.spent_by("ivan/gpt-mini"), replies.to_dict()["replies"][0]["cost"]

({'USD': Money(0.08, 'USD')}, {'USD': Money(0.08, 'USD')}, {'USD': '0.02'})

## 10. Several models, one question

Asking several models is just a search over model endpoints: same pre-flight, same rows, `answers` keyed by model.
Lena bills in rupiah; totals stay per currency.

In [42]:
opinions = await hub.search("What are the main adverse events in phase 3 oncology trials?",
                            sources=["bob/llama-3", "lena/sahabat-ai"]).execute()
opinions.answers

{'bob/llama-3': "I don't have retrieved context for this question, so I can only answer generally: phase 3 oncology trials most often report haematological toxicity, gastrointestinal effects and fatigue.",
 'lena/sahabat-ai': 'In short: generally: haematological toxicity, GI effects, fatigue. No retrieved context was provided.'}

In [43]:
opinions.cost

{'IDR': Money(300.0, 'IDR')}

## 11. Search on every turn

When the question changes faster than the sources, `hub.chat(model, sources=...)` searches before each message. The
same `filter` travels with every turn's search: to the Space where it advertises the field, into this client
otherwise.

In [44]:
live = hub.chat("bob/llama-3", sources=["carol/papers", "erin/trials"]).filter(published_gte="2024-01-01").top(4)
await live.send("How many deaths were reported, and were they related to treatment?")

[3],erin/trials


In [45]:
live.last.results

endpoint,returns,outcome,result,cost,latency
carol/papers,references,returned,2 documents · filtered at the Space,-,501 ms
erin/trials,references,returned,2 documents · filtered at the Space,$0.05,422 ms
source,policy,status,amount,wallet / rail,transaction
erin/trials,stripe_per_request,charged,$0.05,w-erin-stripe-usd,ledger-488586


## 12. Pay per query (MPP, experimental)

An MPP Space answers 402 first. Pre-flight flags it as *will ask to pay* and sends it; the row comes back
`PAYMENT_REQUIRED` with a `Charge`. `approve()` settles it from the Hub wallet and re-sends with the credential.

In [46]:
ledger = await hub.search("settlement events for trial data", sources=["kim/ledger"]).execute()
ledger

Results "settlement events for trial data" (0/1 returned)
  kim/ledger               payment_required                 Payment of $0.05 required to query this endpoint

In [47]:
await ledger.approve()

endpoint,returns,outcome,result,cost,latency
kim/ledger,references,returned,1 document,$0.05,373 ms
source,policy,status,amount,wallet / rail,transaction
kim/ledger,mpp_per_request,charged,$0.05,mpp,0x9315e1a40e1d78ca


## 13. The Aggregator (route proposed)

Two forms, so the SDK is settled whichever way the platform decides. **Documents supplied**: the client already
fanned out; `results.aggregate(model)` sends the view's passages and gets one reranked, cited answer.

In [48]:
await recent.aggregate("bob/llama-3")

[1],carol/papers
[2],olga/trials-assistant
[3],erin/trials
[4],carol/papers
[5],dave/notes


**Sources supplied**: one request, the Aggregator fans out with the client's satellite tokens. Pre-flight still runs
here first, so held sources are never sent. When a Space wants paying anyway, such as an MPP Space answering 402, the
Aggregator cannot pay on your behalf: it returns the per-Space envelopes plus a `job_id`. The same `Charge` and
`TopUp` objects appear on `answer.pending`; pay, then the job resumes with the credentials attached.

In [49]:
answer = await hub.aggregate("settlement events and adverse events", sources=["carol/papers", "kim/ledger"], model="bob/llama-3")
answer

Answer(pending 1, job job-2c1a7cfc)

In [50]:
await answer.approve()                                    # pays the MPP charge from the Hub wallet, resumes the job

Answer(model=bob/llama-3, 297 chars, 5 citations)

## 14. Errors

Exceptions are for misuse and for failures with no remedy inside the flow. Every one is a `SyftHubError`, the message
says what to do, and 401 is never conflated with 403.

In [51]:
from syfthub import SyftHubError, ValidationError, NotFound, InvalidState, NotLoggedIn

for attempt in (lambda: hub.endpoints.get("not-a-path"),
                lambda: hub.endpoints.get("nobody/here"),
                lambda: chat.remove("lena/sahabat-ai"),
                lambda: AsyncHub(options=Options(http_client=MockTransport(World()))).auth.whoami()):
    try:
        await attempt()
    except SyftHubError as e:
        print(f"{type(e).__name__:<16} {e}")

ValidationError  'not-a-path' is not an endpoint path; use 'owner/slug'
NotFound         no endpoint 'nobody/here' on https://hub.syfthub.org
InvalidState     lena/sahabat-ai is not in this chat (asked: bob/llama-3)
NotLoggedIn      Nobody is signed in: await hub.login(username=..., password=...) or hub.login(token=...)


In [52]:
def tree(cls, depth=0):
    print("  " * depth + cls.__name__)
    for sub in cls.__subclasses__():
        tree(sub, depth + 1)
tree(SyftHubError)

SyftHubError
  ConfigurationError
  ValidationError
  AuthError
    NotLoggedIn
  HubError
    NotFound
    HubUnavailable
  SpaceError
    InvoiceError
  AggregatorError
  BudgetExceeded
  InvalidState


## 15. The synchronous twin

Scripts and REPLs without an event loop use `syfthub.Hub`: same methods, no `await`. In the real SDK it is generated
from the async package with `unasync`; the mock stands it in with a proxy.

In [53]:
with syfthub.Hub(options=Options(http_client=MockTransport(World()))) as sync_hub:
    sync_hub.login(username="alice", password="secret")
    r = sync_hub.search("adverse events", sources=["carol/papers", "erin/trials"]).execute()
    print(r.ok_count, r.cost)

2 {'USD': Money(0.05, 'USD')}


## Where this leaves us

- **Three lines to a first result**, free by default; credentials from the environment.
- **Compose is sync and free; network is awaited.** `SearchPlan` is there for whoever wants to look before sending.
- **Namespaces** for growth (`auth`, `endpoints`, `wallets`), two domain verbs at the top (`search`, `chat`), one alias (`login`).
- **Rows, not exceptions**, and the outcome says who stopped the call. Exceptions form one small hierarchy.
- **Immutable results, one mutable room**, with `fork()` to branch it.
- **The transport owns** retries with jitter, the per-Space circuit breaker, token caching, pagination and logging; one `Options` object tunes all of it.
- **Typed everywhere**: enums for closed sets, `Money` for amounts, models for every dict, `to_dict()` for the way out.
- **Headless works**: token login, `wait_paid()`, JSON out.
- **The Aggregator surface is settled** in both forms the platform discussion leaves open.